# Retrain the voice for classical Arabic (private)
Teaches the voice's text reader the sounds it gets wrong (tanwin, shadda, hamza, ص ض ط ظ ق ع, heavy ر) using:
- **ClArTTS**: about 12 h of fully vowelled classical Arabic from the same reader our voice is based on (CC BY 4.0)
- **al-Ḥuṣarī murattal** (optional, `HUSARY_HOURS`): Quran recitation, for articulation. Keep this notebook and its outputs **private**.

Only a small add-on file (the "adapter", about 50 MB) is trained. The voice itself stays the same.

**Settings (right panel):** Accelerator **GPU T4** (one is enough), Internet **On**, keep it **Private**.
Optional: Add-ons → Secrets → `GITHUB_TOKEN` (a GitHub token with repo access). This is needed once the repo is private, and it lets the notebook save the adapter to GitHub as well.

Then **Save Version → Save & Run All**. It takes about 1–1.5 hours. Results are in the Output tab:
- `samples/before_after.mp3`: 8 problem lines, original voice then retrained, so you can judge by ear
- `adapter/adapter_latest.pt`: the trained add-on, plus `adapter/train_log.txt`


In [ ]:
# ---- settings ----
HUSARY_HOURS = 6      # 0 = ClArTTS only
STEPS        = 3000
BATCH        = 8
BRANCH       = "arabic-pipeline"
REPO         = "github.com/mohaammedu172-ux/white-nights-audiobook.git"


In [ ]:
import os, subprocess
tok = ""
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass
url = f"https://x-access-token:{tok}@{REPO}" if tok else f"https://{REPO}"
if not os.path.isdir('/kaggle/working/repo'):
    subprocess.run(['git','clone','--quiet','--depth','1','-b',BRANCH,url,'/kaggle/working/repo'], check=True)
%cd /kaggle/working/repo
!pip -q install chatterbox-tts==0.1.7 --no-deps
!pip -q install s3tokenizer conformer==0.3.2 omegaconf pyloudnorm noisereduce resemble-perth peft "transformers>=5.2,<6" diffusers librosa soundfile safetensors
!pip -q uninstall -y torchao   # Kaggle ships an old torchao that makes the LoRA library refuse to start
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - turn on GPU in settings')


In [ ]:
# 1) training data (about 20-40 min); skipped if it was already built in this session
import os, subprocess
if not os.path.exists('/kaggle/working/data/dataset.pt'):
    subprocess.run(f'python -m ar.train.prep --out /kaggle/working/data --husary-hours {HUSARY_HOURS} --husary-index ar/train/tarteel_index.json', shell=True, check=True)
else:
    print('training data already there')


In [ ]:
# 2) training (about 30-60 min); saves every 500 steps
subprocess.run(f'python -m ar.train.finetune --data /kaggle/working/data/dataset.pt --out /kaggle/working/adapter --steps {STEPS} --batch {BATCH}', shell=True, check=True)


In [ ]:
# 3) before/after listening test
subprocess.run('python -m ar.train.samples --adapter /kaggle/working/adapter/adapter_latest.pt --out /kaggle/working/samples', shell=True, check=True)
from IPython.display import Audio, display
display(Audio(filename='/kaggle/working/samples/before_after.mp3'))


In [ ]:
# 4) keep a copy on GitHub (only if GITHUB_TOKEN is set): branch voice-adapter
import shutil, datetime
if tok:
    d='/kaggle/working/push'; shutil.rmtree(d, ignore_errors=True); os.makedirs(d+'/adapters')
    stamp=datetime.datetime.utcnow().strftime('%Y%m%d-%H%M')
    shutil.copy('/kaggle/working/adapter/adapter_latest.pt', f'{d}/adapters/adapter_{stamp}.pt')
    shutil.copy('/kaggle/working/adapter/train_log.txt', f'{d}/adapters/train_log_{stamp}.txt')
    shutil.copy('/kaggle/working/samples/before_after.mp3', f'{d}/adapters/before_after_{stamp}.mp3')
    cmds = f'''cd {d} && git init -q -b voice-adapter && git remote add origin {url} &&
      (git fetch -q --depth 1 origin voice-adapter && git reset -q --soft FETCH_HEAD || true) &&
      git add adapters && git -c user.name=kaggle -c user.email=kaggle@users.noreply.github.com commit -qm "Voice adapter {stamp}" &&
      git push -q origin voice-adapter'''
    print(subprocess.run(cmds, shell=True, capture_output=True, text=True).stderr[-500:] or 'saved to GitHub branch voice-adapter')
else:
    print('No GITHUB_TOKEN secret: download adapter/adapter_latest.pt from the Output tab instead.')
